# 4 · Die Kampagne: Wen schreiben wir an?
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Päckli & Co. hat die Kampagne an 9'000 zufällig gewählten Kunden getestet (Pilot): je 3'000 ohne E-Mail (`control`),
mit E-Mail (`email`) und mit E-Mail und 10 % Rabatt (`email_discount`). Jetzt entscheidet ihr für die übrigen
11'000 Kunden: **Wen schreiben wir an, und mit welchem Angebot?**

**Zahlen:** Marge 40 %, Rabatt 10 %, E-Mail CHF 0.10. Wer sich abmeldet, ist für künftige E-Mails verloren.
**Preis:** höchster zusätzlicher Gewinn der Kampagne.

**Pflicht:** Schritte 3 bis 5. Schritte 1 und 2 zeigt die Kursleitung live. **Optional:** E1.

**Probiert es selbst.** Kommt ihr nicht weiter, fragt Gemini (✦ oben rechts) oder einen anderen KI-Assistenten eurer Wahl.
Aber: Ihr müsst jede Zeile eurer Lösung verstehen und erklären können.

**Jeder Schritt:** Code in die leere Zelle › ▶ › **Plausibel?** prüfen. Die Lösung gibt es nach der Übung.
Braucht ein Schritt das Ergebnis eines früheren, der nicht geklappt hat: **▶ Aufholen** ausführen.

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, sys, urllib.request
try:
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")   # immer die neuste Version
except OSError:
    if not os.path.exists("kurs.py"): raise
sys.modules.pop("kurs", None)   # neu laden, falls schon geladen
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

In [ ]:
#@title ▶ Vorbereitet: Churn-Risiko und Pilot-Ergebnis laden
churn_scores = catch_up("churn_scores", eigene=False)
pilot = catch_up("pilot_ergebnis", eigene=False).merge(churn_scores[["customer_id", "churn_risiko"]], on="customer_id")
pilot["risiko_gruppe"] = pd.cut(pilot["churn_risiko"], [0, 0.5, 0.7, 0.9, 1.0], include_lowest=True,
                                labels=["1 unter 50 %", "2 50 bis 70 %", "3 70 bis 90 %", "4 über 90 %"])
pilot.head()

## Teil A · Live mit der Kursleitung

### 1 Lohnt sich der Rabatt? Break-even
**Ziel:** Um wie viel muss die Kaufwahrscheinlichkeit mit Rabatt steigen, damit er sich lohnt?

**Aufgabe:** Marge 40 %, Rabatt 10 % auf den Listenpreis. Um welchen Faktor muss die Kaufwahrscheinlichkeit mit Rabatt mindestens steigen, damit der erwartete Gewinn nicht sinkt? Rechne es in Python für eine treue Kundin (kauft sowieso mit 60 %) und einen gefährdeten Kunden (5 %) durch.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Treue Kundin: 80 %, kaum möglich. Gefährdeter Kunde: 6.7 %, machbar. Rabatt lohnt sich nur bei Kunden, die ohne ihn kaum kaufen.

### 2 Der Pilot: Was hat er gebracht?
**Ziel:** Pro Gruppe: Kaufrate und Gewinn pro Kunde. `profit` = Umsatz × 40 % − Rabatt − CHF 0.10 − erwarteter Abmelde-Verlust.

**Aufgabe:** Zeige für den DataFrame pilot pro arm: Anzahl Kunden, Kaufrate (Mittelwert von purchased) und Ø Gewinn (Mittelwert von profit).

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Der Rabatt hat die höchste Kaufrate. Hat er auch den höchsten Gewinn? Es zählt der Gewinn **gegenüber der Kontrollgruppe**: Viele hätten auch ohne E-Mail gekauft.

## Teil B · Eure Kampagne

### 3 Bei wem wirkt die Kampagne?
**Ziel:** Der Pilot, aufgeteilt nach Churn-Risiko: Wo bringt die E-Mail Gewinn, wo kostet sie?

**Aufgabe:** Berechne im DataFrame pilot den Ø profit pro risiko_gruppe und arm (Pivot-Tabelle: Zeilen risiko_gruppe, Spalten arm). Ziehe in jeder Zeile den Wert der Spalte control ab und zeige das Ergebnis für email und email_discount als Balkendiagramm mit einer Linie bei 0.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Sichere Käufer (tiefes Risiko) kaufen sowieso: Der Rabatt kostet dort nur. Verlorene (sehr hohes Risiko) lesen keine E-Mail. Wo sind die Überzeugbaren?

### 4 Eure Entscheidung
Regler einstellen und ausführen, so oft ihr wollt. Die Hochrechnung stammt aus dem Pilot: Sie ist eine Schätzung, keine Garantie.

In [ ]:
#@title 🎚️ Eure Entscheidung: Regler einstellen, dann ▶
min_risiko = 0.0  #@param {type:"slider", min:0, max:0.95, step:0.05}
max_risiko = 1.0  #@param {type:"slider", min:0.05, max:1, step:0.05}
angebot = "email"  #@param ["email", "email_discount"]
zielgruppe = churn_scores[churn_scores["churn_risiko"].between(min_risiko, max_risiko)
                          & ~churn_scores["customer_id"].isin(pilot["customer_id"])]
p = pilot[pilot["churn_risiko"].between(min_risiko, max_risiko)]
print(f"Zielgruppe: {len(zielgruppe)} Kunden mit Churn-Risiko {min_risiko:.0%} bis {max_risiko:.0%}, Angebot: {angebot}")
if len(zielgruppe) and (p["arm"] == angebot).any() and (p["arm"] == "control").any():
    pro_kunde = p.loc[p["arm"] == angebot, "profit"].mean() - p.loc[p["arm"] == "control", "profit"].mean()
    print(f"Im Pilot brachte das in dieser Gruppe {pro_kunde:+.2f} CHF pro Kunde. Hochgerechnet auf die {0.8 * len(zielgruppe):.0f}, die die E-Mail bekommen (20 % bleiben Kontrollgruppe): {pro_kunde * 0.8 * len(zielgruppe):+.0f} CHF.")
else:
    print("⚠️ In diesem Bereich gibt es keine Kunden. Regler anpassen.")
ax = churn_scores["churn_risiko"].plot.hist(bins=40, color="lightgray", figsize=(7, 2.5), title="Alle Kunden nach Churn-Risiko")
ax.axvspan(min_risiko, max_risiko, color="gold", alpha=0.4); ax.set_xlabel("Churn-Risiko (gelb: eure Zielgruppe)"); plt.show()

### ▶ 5 Abgabe (Pflicht)
Eine Person pro Team reicht. Abgeben dürft ihr mehrmals, es zählt die letzte Abgabe. Colab fragt nach Google Drive: euer Konto wählen und alles zulassen. Klappt es nicht: die drei Werte in den Kurs-Chat schreiben.

In [ ]:
save(pd.DataFrame([{"min_risiko": min_risiko, "max_risiko": max_risiko, "angebot": angebot}]), "kampagne")
print(f"Eure Kampagne: Churn-Risiko {min_risiko:.0%} bis {max_risiko:.0%}, Angebot {angebot}")

## Erweiterung (optional)

### E1 E-Mails schreiben lassen (optional)
**Ziel:** Personalisierung im grossen Stil, und wo sie schiefgeht. Wird nicht bewertet.

**Aufgabe:** Lade die Empfehlungen mit catch_up("empfehlungen", eigene=False) und die Produktnamen mit pd.read_parquet("daten/products.parquet"). Zeige für 3 Kunden aus zielgruppe die Produktnamen (description) ihrer Empfehlungen. Schreibe dann für jeden eine kurze, persönliche E-Mail von Päckli & Co. mit diesen drei Produkten.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Würdet ihr die E-Mail öffnen? Erfindet die KI Rabatte oder Produkte?